# 08 - Multi-Book Risk Books, VaR, and Physical Context

Purpose:
- extend the downstream risk-book layer from a single benchmark Brent futures book to a **parallel multi-book evaluation**;
- keep `01`–`07` unchanged and preserve the Gold alarm construction exactly as built;
- evaluate Brent futures, WTI Cushing spot, and Henry Hub spot under one common risk template;
- attach physical interpretation layers for crude and gas without letting weekly inventory data enter daily P&L.

Core design:
- **Signal timeline and book timeline are separated.** The Gold alarm remains fixed from prior notebooks.
- **Each book is prepared independently and then aligned via per-book inner join to the signal timeline.**
- **Weekly stocks / storage are physical context layers only.** They never enter `R_book`, VaR, or lead-time event construction.


## Reader Orientation

This notebook starts the parallel multi-book evaluation without reopening the signal-design question. The Gold alarm has already been built and validated. The task here is narrower:

1. build several proxy books under one consistent template;
2. test whether the same Gold alarm is operationally useful across those books;
3. preserve physical-market interpretation by attaching last-known weekly physical context to the relevant spot books.

Prerequisite outputs: this notebook expects Step 02, Step 04, and Step 06 processed outputs to exist under `data/processed/`. If they are not present in this project directory yet, run `01`, `02`, `04`, and `06` first.


In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd()
PROCESSED_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'outputs' / 'step08_riskbook_var_stress'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
market_vars_path = PROCESSED_DIR / 'market_vars_core.parquet'
if not market_vars_path.exists():
    market_vars_path = PROCESSED_DIR / 'market_vars.parquet'

alarm_frame_path = PROCESSED_DIR / 'gold_alarm_frame.parquet'
market_vars = pd.read_parquet(market_vars_path)
alarm_frame = pd.read_parquet(alarm_frame_path)

required_alarm_cols = ['gold_alarm', 'alarm_score', 'dashboard_state']
missing_alarm = [c for c in required_alarm_cols if c not in alarm_frame.columns]
if missing_alarm:
    raise ValueError(f'Missing required alarm columns: {missing_alarm}')

print('Market vars:', market_vars_path, market_vars.shape)
print('Alarm frame:', alarm_frame_path, alarm_frame.shape)
print('Signal date range:', alarm_frame.index.min(), 'to', alarm_frame.index.max())

## Parameter Rationale

All books share the same risk template so that comparisons remain interpretable.

- `NAV0 = 100`: normalized starting NAV.
- `VAR_CONFIDENCE = 0.95`: standard one-day VaR confidence level.
- `VAR_WINDOW = 250`: one trading year of prior returns for historical simulation.
- `REALIZED_VOL_WINDOW = 20`: one trading month for current realized volatility.
- `DRAWDOWN_EVENT_LEVEL = -0.05`: common drawdown threshold across books.
- `BOOKS`: Brent futures, WTI Cushing spot, and Henry Hub spot.

Direction convention: all books are treated as **long physical inventory / merchant exposure**. Price declines therefore represent risk to the book.


In [ ]:
NAV0 = 100.0
VAR_CONFIDENCE = 0.95
VAR_ALPHA = 1 - VAR_CONFIDENCE
VAR_WINDOW = 250
REALIZED_VOL_WINDOW = 20
DRAWDOWN_EVENT_LEVEL = -0.05
TRADING_DAYS = 252

BOOKS = {
    'Brent_futures': {
        'label': 'Brent futures',
        'book_type': 'futures',
        'price_source': 'Step 02 market_vars r_Brent',
        'context_type': None,
    },
    'WTI_Cushing_spot': {
        'label': 'WTI Cushing spot',
        'book_type': 'spot_proxy',
        'price_source': 'FRED / EIA WTI Cushing spot proxy',
        'context_type': 'crude_stocks',
    },
    'Henry_Hub_spot': {
        'label': 'Henry Hub spot',
        'book_type': 'spot_proxy',
        'price_source': 'FRED Henry Hub spot proxy',
        'context_type': 'gas_storage',
    },
}

## Helper Functions

Two types of helper are needed:

1. **book-construction helpers** for prices, returns, VaR, and drawdowns;
2. **weekly-context helpers** that preserve the distinction between `week_end_date` and `release_date`.

The weekly series are shifted onto `release_date` and only then carried forward as last-known context. This avoids using unpublished physical information in daily risk evaluation.


In [ ]:
def fetch_fred_series(series_id: str, retries: int = 3, timeout: int = 60) -> pd.Series:
    import io
    import time
    import requests

    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, timeout=timeout, headers={'User-Agent': 'Mozilla/5.0'})
            response.raise_for_status()
            df = pd.read_csv(io.StringIO(response.text))
            if df.shape[1] != 2:
                raise ValueError(f'Unexpected FRED format for {series_id}: {df.columns.tolist()}')
            df.columns = ['Date', series_id]
            df['Date'] = pd.to_datetime(df['Date'])
            df[series_id] = pd.to_numeric(df[series_id], errors='coerce')
            return df.set_index('Date')[series_id].sort_index()
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(2 * attempt)
    raise RuntimeError(f'Failed to download FRED series {series_id} after {retries} attempts') from last_error


def flatten_weekly_eia_table(url: str, value_name: str) -> pd.DataFrame:
    tables = pd.read_html(url)
    if len(tables) < 5:
        raise ValueError(f'Unexpected EIA table structure for {url}')
    table = tables[4].copy()
    rows = []
    for _, row in table.iterrows():
        year_month = row.iloc[0]
        if pd.isna(year_month):
            continue
        year = str(year_month).split('-')[0]
        for wk in range(1, 6):
            date_col = ('Week ' + str(wk), 'End Date')
            value_col = ('Week ' + str(wk), 'Value')
            if date_col not in table.columns or value_col not in table.columns:
                continue
            end_date = row[date_col]
            value = row[value_col]
            if pd.isna(end_date) or pd.isna(value):
                continue
            end_date = pd.to_datetime(f'{year}/{end_date}', format='%Y/%m/%d', errors='coerce')
            if pd.isna(end_date):
                continue
            rows.append({'week_end_date': end_date, value_name: float(value)})
    out = pd.DataFrame(rows).sort_values('week_end_date').drop_duplicates('week_end_date')
    out['wow_change'] = out[value_name].diff()
    out['wow_pct_change'] = out[value_name].pct_change()
    out['inventory_zscore'] = (out[value_name] - out[value_name].mean()) / out[value_name].std()
    out['release_date'] = out['week_end_date'] + pd.Timedelta(days=5)
    return out


def build_daily_context(weekly_df: pd.DataFrame, date_index: pd.DatetimeIndex, prefix: str) -> pd.DataFrame:
    context = weekly_df.set_index('release_date')[[col for col in weekly_df.columns if col != 'release_date']].copy()
    context = context.add_prefix(prefix)
    context = context.reindex(date_index).ffill()
    return context


def build_riskbook_from_returns(book_name: str, returns: pd.Series, alarm_frame: pd.DataFrame) -> pd.DataFrame:
    returns = returns.dropna().rename('R_book')
    riskbook = pd.DataFrame(index=returns.index)
    riskbook['R_book'] = returns
    riskbook['nav'] = NAV0 * (1 + riskbook['R_book']).cumprod()
    riskbook['drawdown'] = riskbook['nav'] / riskbook['nav'].cummax() - 1
    riskbook['realized_vol_20d'] = riskbook['R_book'].rolling(REALIZED_VOL_WINDOW).std() * np.sqrt(TRADING_DAYS)

    hist_input = riskbook['R_book'].shift(1)
    riskbook['hs_var_return'] = hist_input.rolling(VAR_WINDOW).quantile(VAR_ALPHA)
    riskbook['hs_es_return'] = hist_input.rolling(VAR_WINDOW).apply(
        lambda x: x[x <= np.quantile(x, VAR_ALPHA)].mean(), raw=True
    )
    riskbook['var_breach'] = (riskbook['R_book'] < riskbook['hs_var_return']).astype(int)
    riskbook['excess_loss_over_var'] = riskbook['hs_var_return'] - riskbook['R_book']
    riskbook = riskbook.join(alarm_frame[required_alarm_cols], how='inner')
    riskbook['book'] = book_name
    return riskbook


def kupiec_summary(riskbook: pd.DataFrame) -> pd.DataFrame:
    coverage_sample = riskbook.dropna(subset=['hs_var_return']).copy()
    n_obs = len(coverage_sample)
    n_breaches = int(coverage_sample['var_breach'].sum())
    p_hat = n_breaches / n_obs if n_obs else np.nan
    n_non_breaches = n_obs - n_breaches if n_obs else 0
    if n_obs and 0 < p_hat < 1:
        lr_stat = -2 * (
            n_breaches * np.log(VAR_ALPHA / p_hat)
            + n_non_breaches * np.log((1 - VAR_ALPHA) / (1 - p_hat))
        )
        kupiec_pvalue = float(stats.chi2.sf(lr_stat, df=1))
    else:
        lr_stat = np.nan
        kupiec_pvalue = np.nan

    return pd.DataFrame([{
        'book': riskbook['book'].iloc[0],
        'var_confidence': VAR_CONFIDENCE,
        'expected_breach_rate': VAR_ALPHA,
        'observed_breach_rate': p_hat,
        'breach_count': n_breaches,
        'sample_days': n_obs,
        'max_drawdown': riskbook['drawdown'].min(),
        'lr_statistic': lr_stat,
        'p_value': kupiec_pvalue,
        'reject_h0_5pct': bool(kupiec_pvalue < 0.05) if pd.notna(kupiec_pvalue) else False,
    }])

## Build Book Price Legs And Weekly Physical Context

The books are prepared independently before they are aligned with the Gold signal timeline.

- Brent returns come from Step 02 directly.
- WTI and Henry Hub are downloaded as daily price observations and converted to log returns.
- Weekly crude stocks and gas storage are flattened from EIA history pages, mapped to `release_date`, and only then carried forward as last-known context.


In [ ]:
book_inputs = {}

# Brent futures book from the existing cleaned market variables
if 'r_Brent' not in market_vars.columns:
    raise ValueError('Expected r_Brent in Step 02 output for Brent book.')
book_inputs['Brent_futures'] = {
    'returns': market_vars['r_Brent'].dropna(),
    'prices': None,
    'context_daily': pd.DataFrame(index=market_vars.index),
}

# WTI Cushing spot proxy
wti_price = fetch_fred_series('DCOILWTICO').rename('WTI_Cushing_spot_price')
wti_price = wti_price.replace('.', np.nan).astype(float).dropna()
book_inputs['WTI_Cushing_spot'] = {
    'returns': np.log(wti_price / wti_price.shift(1)).dropna().rename('R_book'),
    'prices': wti_price,
    'context_daily': pd.DataFrame(index=wti_price.index),
}

# Henry Hub spot proxy — use FRED daily spot series when available
hh_price = None
hh_candidates = ['DHHNGSP', 'MHHNGSP']
for candidate in hh_candidates:
    try:
        series = fetch_fred_series(candidate)
        series = series.replace('.', np.nan).astype(float).dropna()
        if len(series) > 100:
            hh_price = series.rename('Henry_Hub_spot_price')
            hh_series_id = candidate
            break
    except Exception:
        continue
if hh_price is None:
    raise RuntimeError('Could not download a Henry Hub spot series from FRED candidates.')
book_inputs['Henry_Hub_spot'] = {
    'returns': np.log(hh_price / hh_price.shift(1)).dropna().rename('R_book'),
    'prices': hh_price,
    'context_daily': pd.DataFrame(index=hh_price.index),
}

# Crude physical context for WTI
cushing_weekly = flatten_weekly_eia_table(
    'https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?f=W&n=PET&s=W_EPC0_SAX_YCUOK_MBBL',
    'inventory_level'
)
us_crude_weekly = flatten_weekly_eia_table(
    'https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?f=W&n=PET&s=WCESTUS1',
    'inventory_level'
)
book_inputs['WTI_Cushing_spot']['context_daily'] = pd.concat([
    build_daily_context(cushing_weekly, book_inputs['WTI_Cushing_spot']['returns'].index, 'cushing_'),
    build_daily_context(us_crude_weekly, book_inputs['WTI_Cushing_spot']['returns'].index, 'us_crude_'),
], axis=1)

# Gas storage context for Henry Hub
lower48_storage_weekly = flatten_weekly_eia_table(
    'https://www.eia.gov/dnav/ng/hist/nw2_epg0_swo_r48_bcfw.htm',
    'storage_level'
)
book_inputs['Henry_Hub_spot']['context_daily'] = build_daily_context(
    lower48_storage_weekly,
    book_inputs['Henry_Hub_spot']['returns'].index,
    'gas_storage_'
)

print('WTI FRED observations:', len(wti_price))
print('Henry Hub series id used:', hh_series_id, 'observations:', len(hh_price))
print('Cushing weekly rows:', len(cushing_weekly))
print('US crude weekly rows:', len(us_crude_weekly))
print('Lower-48 gas storage weekly rows:', len(lower48_storage_weekly))

### Why Weekly Context Stays Outside `R_book`

The weekly crude and gas series are useful physical context, but they are not daily tradable return legs. They are therefore treated as **last-known released state variables** only.

This means:
- no interpolation of weekly series into daily P&L;
- no use of unpublished week-end values before the stated release date;
- no inventory values inside VaR or lead-time event definitions.


In [ ]:
context_preview = {
    'WTI_cushing_context_head': book_inputs['WTI_Cushing_spot']['context_daily'].dropna().head(),
    'HenryHub_storage_context_head': book_inputs['Henry_Hub_spot']['context_daily'].dropna().head(),
}
context_preview['WTI_cushing_context_head']

## Build Parallel Risk Books

Each book is constructed from its own return series and then aligned to the existing Gold alarm outputs via **per-book inner join**. This preserves the signal sample while allowing each book to keep its own valid evaluation window.


In [ ]:
riskbooks = {}
book_rows = []
kupiec_rows = []

for book_name, spec in BOOKS.items():
    rb = build_riskbook_from_returns(book_name, book_inputs[book_name]['returns'], alarm_frame)
    context = book_inputs[book_name]['context_daily']
    if not context.empty:
        rb = rb.join(context, how='left')
    riskbooks[book_name] = rb

    summary = kupiec_summary(rb)
    summary['label'] = spec['label']
    summary['book_type'] = spec['book_type']
    book_rows.append(summary)
    kupiec_rows.append(summary[[
        'book', 'expected_breach_rate', 'observed_breach_rate',
        'breach_count', 'sample_days', 'lr_statistic', 'p_value', 'reject_h0_5pct'
    ]])

book_comparison_summary = pd.concat(book_rows, ignore_index=True)
book_var_backtest_comparison = pd.concat(kupiec_rows, ignore_index=True)
book_comparison_summary[[
    'book', 'label', 'book_type', 'sample_days', 'observed_breach_rate',
    'breach_count', 'max_drawdown', 'p_value', 'reject_h0_5pct'
]]

## Visual Comparison

The plots below keep the single-book diagnostics but put them side by side so the books can be compared under the same template.


In [ ]:
fig, axes = plt.subplots(len(riskbooks), 3, figsize=(15, 4 * len(riskbooks)), sharex=False)
if len(riskbooks) == 1:
    axes = np.array([axes])

for row_idx, (book_name, rb) in enumerate(riskbooks.items()):
    ax_nav, ax_dd, ax_var = axes[row_idx]
    label = BOOKS[book_name]['label']

    rb['nav'].plot(ax=ax_nav, linewidth=0.9)
    ax_nav.set_title(f'{label} NAV')
    ax_nav.set_ylabel('NAV')

    rb['drawdown'].plot(ax=ax_dd, linewidth=0.9)
    ax_dd.axhline(DRAWDOWN_EVENT_LEVEL, color='red', linestyle='--', linewidth=0.8)
    ax_dd.set_title(f'{label} drawdown')
    ax_dd.set_ylabel('Drawdown')

    rb[['R_book', 'hs_var_return']].plot(ax=ax_var, linewidth=0.8)
    ax_var.scatter(
        rb.index[rb['var_breach'] == 1],
        rb.loc[rb['var_breach'] == 1, 'R_book'],
        s=10, color='red', label='VaR breach'
    )
    ax_var.set_title(f'{label} return vs HS VaR')
    ax_var.set_ylabel('Return')
    ax_var.legend(fontsize=7)

plt.tight_layout()

## Gold-Triggered Stress Scenarios

The original Brent scenarios remain the core template. They are retained here for dashboard action logic and book comparison, not as optimized scenario sets for each new market.

For WTI and Henry Hub, the scenarios should be read as **macro stress narratives** rather than precise physical revaluation recipes.


In [ ]:
stress_scenarios = pd.DataFrame([
    {
        'scenario': 'Risk-off growth shock',
        'r_Brent': -0.08,
        'r_WTI': -0.09,
        'r_HenryHub': -0.06,
        'r_Gold': 0.03,
        'r_DXY': 0.02,
        'd_VIX': 12.0,
        'd_US10Y': -0.20,
        'description': 'Demand shock: crude and gas weaken while gold, USD, and VIX rise.',
    },
    {
        'scenario': 'Geopolitical inflation shock',
        'r_Brent': 0.10,
        'r_WTI': 0.09,
        'r_HenryHub': 0.07,
        'r_Gold': 0.04,
        'r_DXY': 0.01,
        'd_VIX': 8.0,
        'd_US10Y': 0.10,
        'description': 'Supply-risk shock: energy prices and gold rise together.',
    },
    {
        'scenario': 'Liquidity liquidation shock',
        'r_Brent': -0.06,
        'r_WTI': -0.07,
        'r_HenryHub': -0.05,
        'r_Gold': -0.04,
        'r_DXY': 0.03,
        'd_VIX': 15.0,
        'd_US10Y': -0.30,
        'description': 'Cash scramble: risk assets and haven assets sell off together.',
    },
    {
        'scenario': 'Monetary tightening shock',
        'r_Brent': -0.03,
        'r_WTI': -0.04,
        'r_HenryHub': -0.03,
        'r_Gold': -0.05,
        'r_DXY': 0.04,
        'd_VIX': 5.0,
        'd_US10Y': 0.35,
        'description': 'Higher real-rate and USD shock pressures commodity books.',
    },
])

for book_name in ['Brent_futures', 'WTI_Cushing_spot', 'Henry_Hub_spot']:
    col_map = {
        'Brent_futures': 'r_Brent',
        'WTI_Cushing_spot': 'r_WTI',
        'Henry_Hub_spot': 'r_HenryHub',
    }
    ret_col = col_map[book_name]
    stress_scenarios[f'{book_name}_book_return'] = stress_scenarios[ret_col]
    stress_scenarios[f'{book_name}_book_pnl_on_nav100'] = stress_scenarios[ret_col] * NAV0

stress_scenarios

## Save Multi-Book Outputs

The saved outputs are deliberately book-agnostic so that Notebook 09 can consume them without re-running any signal logic.


In [ ]:
multi_riskbook = pd.concat(riskbooks.values()).sort_index()

multi_riskbook.to_parquet(PROCESSED_DIR / 'riskbook_var_multi.parquet')
book_comparison_summary.to_csv(OUTPUT_DIR / 'book_comparison_summary.csv', index=False)
book_var_backtest_comparison.to_csv(OUTPUT_DIR / 'book_var_backtest_comparison.csv', index=False)
stress_scenarios.to_csv(OUTPUT_DIR / 'stress_scenarios_multi.csv', index=False)
cushing_weekly.to_csv(OUTPUT_DIR / 'wti_cushing_weekly_context.csv', index=False)
us_crude_weekly.to_csv(OUTPUT_DIR / 'us_crude_weekly_context.csv', index=False)
lower48_storage_weekly.to_csv(OUTPUT_DIR / 'henry_hub_storage_weekly_context.csv', index=False)

print('Saved Step 08 multi-book outputs to:', OUTPUT_DIR)
print('Riskbook parquet:', PROCESSED_DIR / 'riskbook_var_multi.parquet')